In [ ]:
# Tokenek

from selenium import webdriver
import json
import time
driver = webdriver.Chrome()

url = "https://www.kiwi.com/hu/search/results/budapest-magyarorszag/barcelona-spanyolorszag/"
driver.get(url)

print("Oldal betöltve, hálózati forgalom figyelése...")
time.sleep(12)  # várjuk meg a GraphQL hívásokat

logs = driver.get_log("performance")

umbrella_token = None
visitor_id = None
rand_id = None

for entry in logs:
    message = json.loads(entry["message"])["message"]

    if (
        message["method"] == "Network.requestWillBeSent"
        and "graphql" in message["params"]["request"]["url"]
    ):
        headers = message["params"]["request"]["headers"]

        umbrella_token = headers.get("kw-umbrella-token")
        visitor_id = headers.get("kw-skypicker-visitor-uniqid")
        rand_id = headers.get("kw-x-rand-id")

        if umbrella_token or visitor_id:
            print("\n🎯 GraphQL request elkapva!")
            break

driver.quit()

print("\n===== TOKENEK =====")
print("kw-umbrella-token:", umbrella_token)
print("kw-skypicker-visitor-uniqid:", visitor_id)
print("kw-x-rand-id:", rand_id)


In [24]:
# Repjegyek proto

from selenium import webdriver
import json
import time
import requests

# 1. Tokenek megszerzése Seleniummal
print("🚀 Tokenek megszerzése...")
driver = webdriver.Chrome()

url = "https://www.kiwi.com/hu/search/results/budapest-magyarorszag/barcelona-spanyolorszag/"
driver.get(url)

print("⏳ Oldal betöltve, várakozás a GraphQL hívásokra...")
time.sleep(6)

logs = driver.get_log("performance")

umbrella_token = None
visitor_id = None
rand_id = None

for entry in logs:
    message = json.loads(entry["message"])["message"]
    
    if (
        message["method"] == "Network.requestWillBeSent"
        and "graphql" in message["params"]["request"]["url"]
    ):
        headers = message["params"]["request"]["headers"]
        
        umbrella_token = headers.get("kw-umbrella-token")
        visitor_id = headers.get("kw-skypicker-visitor-uniqid")
        rand_id = headers.get("kw-x-rand-id")
        
        if umbrella_token:
            break

driver.quit()

print("\n✅ Tokenek megvannak:")
print(f"  umbrella: {umbrella_token[:50]}..." if umbrella_token else "  umbrella: None")
print(f"  visitor: {visitor_id}")
print(f"  rand_id: {rand_id}")

# 2. GraphQL query - JAVÍTOTT verzió az ItineraryReturn típussal
print("\n🔍 Járatok lekérése...")

graphql_payload = {
    "query": """
    query SearchReturnItinerariesQuery(
      $search: SearchReturnInput
      $filter: ItinerariesFilterInput
      $options: ItinerariesOptionsInput
    ) {
      returnItineraries(search: $search, filter: $filter, options: $options) {
        __typename
        ... on Itineraries {
          metadata {
            itinerariesCount
          }
          itineraries {
            __typename
            ... on ItineraryReturn {
              id
              legacyId
              price {
                amount
              }
              priceEur {
                amount
              }
              duration
              provider {
                name
                code
              }
              outbound {
                id
                duration
                sectorSegments {
                  segment {
                    id
                    source {
                      localTime
                      utcTimeIso
                      station {
                        code
                        name
                        city {
                          name
                        }
                      }
                    }
                    destination {
                      localTime
                      utcTimeIso
                      station {
                        code
                        name
                        city {
                          name
                        }
                      }
                    }
                    carrier {
                      name
                      code
                    }
                    duration
                  }
                }
              }
              inbound {
                id
                duration
                sectorSegments {
                  segment {
                    id
                    source {
                      localTime
                      station {
                        code
                        name
                      }
                    }
                    destination {
                      localTime
                      station {
                        code
                        name
                      }
                    }
                    carrier {
                      name
                      code
                    }
                  }
                }
              }
            }
          }
        }
        ... on AppError {
          error: message
        }
      }
    }
    """,
    "variables": {
        "search": {
            "itinerary": {
                "source": {"ids": ["City:budapest_hu"]},
                "destination": {"ids": ["City:barcelona_es"]}
            },
            "passengers": {
                "adults": 1
            }
        },
        "filter": {
            "transportTypes": ["FLIGHT"],
            "limit": 10
        },
        "options": {
            "currency": "huf",
            "locale": "hu",
            "market": "hu",
            "partner": "skypicker"
        }
    }
}

# Requests használata
headers = {
    "Content-Type": "application/json",
    "kw-umbrella-token": umbrella_token,
    "kw-skypicker-visitor-uniqid": visitor_id,
    "kw-x-rand-id": rand_id,
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.post(
    "https://api.skypicker.com/umbrella/v2/graphql?featureName=SearchReturnItinerariesQuery",
    json=graphql_payload,
    headers=headers,
    timeout=30
)

# Válasz feldolgozása
data = response.json()

print(f"\n✅ Válasz érkezett (status: {response.status_code})")

# Hibakezelés
if "errors" in data:
    print("\n❌ GraphQL hibák:")
    for error in data["errors"]:
        print(f"  - {error['message']}")
    print("\n💾 Hibás válasz mentve: kiwi_error.json")
    with open("kiwi_error.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
elif data.get("data") and data["data"].get("returnItineraries"):
    result = data["data"]["returnItineraries"]
    
    if result["__typename"] == "Itineraries":
        itineraries = result.get("itineraries", [])
        
        print(f"\n🎫 Talált járatok száma: {len(itineraries)}\n")
        
        for i, flight in enumerate(itineraries[:5], 1):
            if flight["__typename"] != "ItineraryReturn":
                continue
                
            price = float(flight["price"]["amount"])
            duration = flight["duration"] // 3600  # óra
            provider = flight["provider"]["name"]
            
            # Outbound (oda)
            outbound = flight["outbound"]["sectorSegments"][0]["segment"]
            dep = outbound["source"]["station"]["code"]
            dep_city = outbound["source"]["station"]["city"]["name"]
            arr = outbound["destination"]["station"]["code"]
            arr_city = outbound["destination"]["station"]["city"]["name"]
            dep_time = outbound["source"]["localTime"]
            carrier = outbound["carrier"]["name"]
            
            # Inbound (vissza)
            inbound = flight["inbound"]["sectorSegments"][0]["segment"]
            ret_time = inbound["source"]["localTime"]
            
            print(f"{i}. {dep_city} ({dep}) ⇄ {arr_city} ({arr})")
            print(f"   Ár: {price:,.0f} HUF | Időtartam: ~{duration}h")
            print(f"   Oda: {dep_time} | Vissza: {ret_time}")
            print(f"   Légitársaság: {carrier} | {provider}")
            print()
    else:
        print(f"\n❌ Hiba: {result.get('error', 'Ismeretlen hiba')}")
else:
    print("\n❌ Nem érkezett adat")

🚀 Tokenek megszerzése...
⏳ Oldal betöltve, várakozás a GraphQL hívásokra...

✅ Tokenek megvannak:
  umbrella: 127023f99e6012d1485bc942f7e4ab6cd3c2727040a9d8e3ef...
  visitor: b07e15c2-b170-4340-9666-ffb0670de46e
  rand_id: 58fe098a3b2f66b7b435a6ed150efcba9766811c

🔍 Járatok lekérése...

✅ Válasz érkezett (status: 200)

🎫 Talált járatok száma: 10

1. Budapest (BUD) ⇄ Barcelona (BCN)
   Ár: 20,109 HUF | Időtartam: ~5h
   Oda: 2026-01-23T17:45:00 | Vissza: 2026-02-04T18:50:00
   Légitársaság: Ryanair | Kiwi.com

2. Budapest (BUD) ⇄ Barcelona (BCN)
   Ár: 21,252 HUF | Időtartam: ~5h
   Oda: 2026-02-25T15:35:00 | Vissza: 2026-03-09T06:25:00
   Légitársaság: Ryanair | Kiwi.com

3. Budapest (BUD) ⇄ Barcelona (BCN)
   Ár: 21,252 HUF | Időtartam: ~5h
   Oda: 2026-02-25T15:35:00 | Vissza: 2026-03-11T09:30:00
   Légitársaság: Ryanair | Kiwi.com

4. Budapest (BUD) ⇄ Barcelona (BCN)
   Ár: 21,668 HUF | Időtartam: ~5h
   Oda: 2026-01-23T17:45:00 | Vissza: 2026-02-01T21:00:00
   Légitársaság: Ryanair

In [29]:
from selenium import webdriver
import json
import time
import requests
import pandas as pd
from datetime import datetime
from typing import Optional, List

def get_kiwi_tokens(headless: bool = False) -> dict:
    """
    Kiwi.com tokenek megszerzése Selenium segítségével.
    
    Args:
        headless: Ha True, háttérben fut a böngésző
        
    Returns:
        Dictionary a tokenekkel: umbrella_token, visitor_id, rand_id
    """
    print("🚀 Tokenek megszerzése...")
    
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument('--headless')
    
    driver = webdriver.Chrome(options=options)
    
    # Egyszerű keresés a tokenekhez (bármelyik útvonal jó)
    url = "https://www.kiwi.com/hu/search/results/budapest-magyarorszag/barcelona-spanyolorszag/"
    driver.get(url)
    
    print("⏳ Várakozás a GraphQL hívásokra...")
    time.sleep(12)
    
    logs = driver.get_log("performance")
    
    umbrella_token = None
    visitor_id = None
    rand_id = None
    
    for entry in logs:
        message = json.loads(entry["message"])["message"]
        
        if (
            message["method"] == "Network.requestWillBeSent"
            and "graphql" in message["params"]["request"]["url"]
        ):
            headers = message["params"]["request"]["headers"]
            
            umbrella_token = headers.get("kw-umbrella-token")
            visitor_id = headers.get("kw-skypicker-visitor-uniqid")
            rand_id = headers.get("kw-x-rand-id")
            
            if umbrella_token:
                break
    
    driver.quit()
    
    print("✅ Tokenek megszerzve\n")
    
    return {
        "umbrella_token": umbrella_token,
        "visitor_id": visitor_id,
        "rand_id": rand_id
    }


def search_flights(
    origin: str,
    destination: str,
    tokens: dict,
    date_from: Optional[str] = None,
    date_to: Optional[str] = None,
    return_from: Optional[str] = None,
    return_to: Optional[str] = None,
    adults: int = 1,
    children: int = 0,
    infants: int = 0,
    limit: int = 50,
    currency: str = "huf",
    locale: str = "hu",
    max_stopovers: Optional[int] = None,
    direct_flights_only: bool = False
) -> pd.DataFrame:
    """
    Kiwi.com járatok keresése.
    
    Args:
        origin: Indulási város/reptér kódja (pl. "budapest_hu", "BUD")
        destination: Célállomás kódja (pl. "barcelona_es", "BCN")
        tokens: Token dict a get_kiwi_tokens()-ból
        date_from: Indulás kezdő dátuma (YYYY-MM-DD)
        date_to: Indulás záró dátuma (YYYY-MM-DD)
        return_from: Visszaút kezdő dátuma (YYYY-MM-DD)
        return_to: Visszaút záró dátuma (YYYY-MM-DD)
        adults: Felnőttek száma
        children: Gyerekek száma
        infants: Csecsemők száma
        limit: Max találatok száma
        currency: Pénznem (huf, eur, usd)
        locale: Nyelv (hu, en)
        max_stopovers: Max átszállások száma (None = bármi)
        direct_flights_only: Csak direkt járatok
        
    Returns:
        Pandas DataFrame a találatokkal
    """
    print(f"🔍 Járatok keresése: {origin} → {destination}")
    
    # City ID formázás (ha csak kód van megadva)
    if ":" not in origin:
        origin_id = f"City:{origin.lower()}"
    else:
        origin_id = origin
        
    if ":" not in destination:
        dest_id = f"City:{destination.lower()}"
    else:
        dest_id = destination
    
    # GraphQL query
    graphql_payload = {
        "query": """
        query SearchReturnItinerariesQuery(
          $search: SearchReturnInput
          $filter: ItinerariesFilterInput
          $options: ItinerariesOptionsInput
        ) {
          returnItineraries(search: $search, filter: $filter, options: $options) {
            __typename
            ... on Itineraries {
              metadata {
                itinerariesCount
              }
              itineraries {
                __typename
                ... on ItineraryReturn {
                  id
                  legacyId
                  price {
                    amount
                  }
                  priceEur {
                    amount
                  }
                  duration
                  pnrCount
                  provider {
                    name
                    code
                  }
                  outbound {
                    id
                    duration
                    sectorSegments {
                      segment {
                        id
                        source {
                          localTime
                          utcTimeIso
                          station {
                            code
                            name
                            city {
                              name
                            }
                            country {
                              code
                            }
                          }
                        }
                        destination {
                          localTime
                          utcTimeIso
                          station {
                            code
                            name
                            city {
                              name
                            }
                            country {
                              code
                            }
                          }
                        }
                        carrier {
                          name
                          code
                        }
                        operatingCarrier {
                          name
                          code
                        }
                        duration
                        code
                      }
                      layover {
                        duration
                      }
                    }
                  }
                  inbound {
                    id
                    duration
                    sectorSegments {
                      segment {
                        id
                        source {
                          localTime
                          utcTimeIso
                          station {
                            code
                            name
                            city {
                              name
                            }
                          }
                        }
                        destination {
                          localTime
                          utcTimeIso
                          station {
                            code
                            name
                            city {
                              name
                            }
                          }
                        }
                        carrier {
                          name
                          code
                        }
                        duration
                      }
                      layover {
                        duration
                      }
                    }
                  }
                  bookingOptions {
                    edges {
                      node {
                        price {
                          amount
                        }
                        bookingUrl
                      }
                    }
                  }
                }
              }
            }
            ... on AppError {
              error: message
            }
          }
        }
        """,
        "variables": {
            "search": {
                "itinerary": {
                    "source": {"ids": [origin_id]},
                    "destination": {"ids": [dest_id]}
                },
                "passengers": {
                    "adults": adults,
                    "children": children,
                    "infants": infants
                }
            },
            "filter": {
                "transportTypes": ["FLIGHT"],
                "limit": limit
            },
            "options": {
                "currency": currency,
                "locale": locale,
                "market": locale,
                "partner": "skypicker"
            }
        }
    }
    
    # Dátumok hozzáadása ha meg vannak adva
    if date_from or date_to:
        outbound_date = {}
        if date_from:
            outbound_date["start"] = f"{date_from}T00:00:00"
        if date_to:
            outbound_date["end"] = f"{date_to}T23:59:59"
        graphql_payload["variables"]["search"]["itinerary"]["outboundDepartureDate"] = outbound_date
    
    if return_from or return_to:
        inbound_date = {}
        if return_from:
            inbound_date["start"] = f"{return_from}T00:00:00"
        if return_to:
            inbound_date["end"] = f"{return_to}T23:59:59"
        graphql_payload["variables"]["search"]["itinerary"]["inboundDepartureDate"] = inbound_date
    
    # Átszállások szűrése
    if direct_flights_only:
        graphql_payload["variables"]["filter"]["maxStopovers"] = 0
    elif max_stopovers is not None:
        graphql_payload["variables"]["filter"]["maxStopovers"] = max_stopovers
    
    # API hívás
    headers = {
        "Content-Type": "application/json",
        "kw-umbrella-token": tokens["umbrella_token"],
        "kw-skypicker-visitor-uniqid": tokens["visitor_id"],
        "kw-x-rand-id": tokens["rand_id"],
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    
    response = requests.post(
        "https://api.skypicker.com/umbrella/v2/graphql?featureName=SearchReturnItinerariesQuery",
        json=graphql_payload,
        headers=headers,
        timeout=30
    )
    
    data = response.json()
    
    # Hibakezelés
    if "errors" in data:
        print("\n❌ GraphQL hibák:")
        for error in data["errors"]:
            print(f"  - {error['message']}")
        return pd.DataFrame()
    
    if not data.get("data") or not data["data"].get("returnItineraries"):
        print("❌ Nem érkezett adat")
        return pd.DataFrame()
    
    result = data["data"]["returnItineraries"]
    
    if result["__typename"] != "Itineraries":
        print(f"❌ Hiba: {result.get('error', 'Ismeretlen hiba')}")
        return pd.DataFrame()
    
    itineraries = result.get("itineraries", [])
    print(f"✅ {len(itineraries)} járat találva\n")
    
    # DataFrame építése
    flights_data = []
    
    for flight in itineraries:
        if flight["__typename"] != "ItineraryReturn":
            continue
        
        # Alapadatok
        price = float(flight["price"]["amount"])
        price_eur = float(flight["priceEur"]["amount"])
        total_duration_hours = flight["duration"] / 3600
        
        # Outbound (oda)
        outbound = flight["outbound"]
        out_segments = outbound["sectorSegments"]
        out_first = out_segments[0]["segment"]
        out_last = out_segments[-1]["segment"]
        
        dep_airport = out_first["source"]["station"]["code"]
        dep_city = out_first["source"]["station"]["city"]["name"]
        dep_time = out_first["source"]["localTime"]
        
        arr_airport = out_last["destination"]["station"]["code"]
        arr_city = out_last["destination"]["station"]["city"]["name"]
        arr_time = out_last["destination"]["localTime"]
        
        out_duration_hours = outbound["duration"] / 3600
        out_stops = len(out_segments) - 1
        
        # Carriers (légitársaságok)
        out_carriers = list(set([seg["segment"]["carrier"]["name"] for seg in out_segments]))
        
        # Inbound (vissza)
        inbound = flight["inbound"]
        in_segments = inbound["sectorSegments"]
        in_first = in_segments[0]["segment"]
        in_last = in_segments[-1]["segment"]
        
        ret_dep_time = in_first["source"]["localTime"]
        ret_arr_time = in_last["destination"]["localTime"]
        ret_duration_hours = inbound["duration"] / 3600
        ret_stops = len(in_segments) - 1
        
        in_carriers = list(set([seg["segment"]["carrier"]["name"] for seg in in_segments]))
        
        # Booking URL
        booking_url = None
        if flight.get("bookingOptions") and flight["bookingOptions"]["edges"]:
            booking_url = flight["bookingOptions"]["edges"][0]["node"].get("bookingUrl")
        
        flights_data.append({
            "id": flight["id"],
            "price_huf": price,
            "price_eur": price_eur,
            "provider": flight["provider"]["name"],
            
            # Outbound
            "dep_city": dep_city,
            "dep_airport": dep_airport,
            "dep_time": dep_time,
            "arr_city": arr_city,
            "arr_airport": arr_airport,
            "arr_time": arr_time,
            "out_duration_h": round(out_duration_hours, 1),
            "out_stops": out_stops,
            "out_carriers": ", ".join(out_carriers),
            
            # Inbound
            "ret_dep_time": ret_dep_time,
            "ret_arr_time": ret_arr_time,
            "ret_duration_h": round(ret_duration_hours, 1),
            "ret_stops": ret_stops,
            "ret_carriers": ", ".join(in_carriers),
            
            # Összesített
            "total_duration_h": round(total_duration_hours, 1),
            "booking_url": booking_url
        })
    
    df = pd.DataFrame(flights_data)
    
    # Dátum oszlopok konvertálása
    date_cols = ["dep_time", "arr_time", "ret_dep_time", "ret_arr_time"]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col])
    
    # Rendezés ár szerint
    df = df.sort_values("price_huf").reset_index(drop=True)
    
    return df


# ===== PÉLDA HASZNÁLAT =====

if __name__ == "__main__":
    # 1. Tokenek megszerzése (csak egyszer kell)
    tokens = get_kiwi_tokens(headless=False)
    
    # 2. Járatok keresése
    df = search_flights(
        origin="budapest_hu",
        destination="barcelona_es",
        tokens=tokens,
        date_from="2026-01-20",  # Indulás legkorábbi dátuma
        date_to="2026-01-30",    # Indulás legkésőbbi dátuma
        return_from="2026-02-01",  # Visszaút legkorábbi dátuma
        return_to="2026-02-15",    # Visszaút legkésőbbi dátuma
        adults=1,
        limit=50,
        direct_flights_only=False,  # True = csak direkt
        #max_stopovers=1  # Max 1 átszállás
    )
    
    # 3. Eredmények megjelenítése
    if not df.empty:
        print("\n📊 TOP 10 LEGOLCSÓBB JÁRAT:\n")
        
        # Szép formázás
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', 30)
        
        display_df = df.head(10).copy()
        display_df["dep_time"] = display_df["dep_time"].dt.strftime("%Y-%m-%d %H:%M")
        display_df["ret_dep_time"] = display_df["ret_dep_time"].dt.strftime("%Y-%m-%d %H:%M")
        
        print(display_df[[
            "price_huf", "dep_city", "dep_time", "out_stops", 
            "ret_dep_time", "ret_stops", "out_carriers"
        ]].to_string(index=False))

🚀 Tokenek megszerzése...
⏳ Várakozás a GraphQL hívásokra...
✅ Tokenek megszerzve

🔍 Járatok keresése: budapest_hu → barcelona_es
✅ 50 járat találva


📊 TOP 10 LEGOLCSÓBB JÁRAT:

 price_huf dep_city         dep_time  out_stops     ret_dep_time  ret_stops out_carriers
20109.0002 Budapest 2026-01-23 17:45          0 2026-02-04 18:50          0      Ryanair
21667.9999 Budapest 2026-01-23 17:45          0 2026-02-01 21:00          0      Ryanair
22445.0000 Budapest 2026-01-23 17:45          0 2026-02-03 12:35          0      Ryanair
22833.9999 Budapest 2026-01-23 17:45          0 2026-02-02 20:40          0      Ryanair
22833.9999 Budapest 2026-01-23 17:45          0 2026-02-09 06:25          0      Ryanair
22833.9999 Budapest 2026-01-23 17:45          0 2026-02-10 12:35          0      Ryanair
23610.9999 Budapest 2026-01-23 17:45          0 2026-02-02 06:25          0      Ryanair
23999.0000 Budapest 2026-01-23 17:45          0 2026-02-11 18:50          0      Ryanair
24217.4843 Budapest 2